# SLM autónomo en Google Colab - ejecución paso a paso

Ejecuta las celdas en orden. Cada comando se imprime antes de ejecutarse y su salida se muestra en vivo. El notebook descarga el ZIP público del repositorio; no usa el selector de archivos ni guarda secretos.

> Configura **GPU** en `Entorno de ejecución > Cambiar tipo de entorno de ejecución` antes de comenzar.

## Paso 1 - Diagnóstico y GPU

Verifica que Colab asignó una GPU NVIDIA con CUDA. El notebook no continuará en CPU.

In [ ]:
import os
import platform
import subprocess
import sys

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTHONUTF8', '1')

print('Python:', sys.version.split()[0])
print('Sistema:', platform.platform())
print('Directorio inicial:', os.getcwd())
print('\n>>> nvidia-smi')
subprocess.run(['nvidia-smi'], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA no está disponible. Activa GPU en Colab y reinicia el runtime.')
print('\nPyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('✓ GPU lista')

## Paso 2 - Definir el ejecutor con salida en vivo

Esta función imprime el comando y transmite cada línea de salida, incluyendo errores. Las siguientes etapas la reutilizan.

In [ ]:
from pathlib import Path
import shlex

def ejecutar(comando, cwd=None):
    """Ejecuta un comando y muestra stdout/stderr en tiempo real."""
    print('\n>>>', shlex.join([str(x) for x in comando]))
    if cwd:
        print('cwd:', cwd)
    entorno = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'PYTHONUTF8': '1'}
    proceso = subprocess.Popen(
        [str(x) for x in comando], cwd=cwd, env=entorno,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    )
    for linea in iter(proceso.stdout.readline, ''):
        print(linea, end='')
    codigo = proceso.wait()
    if codigo != 0:
        raise RuntimeError(f'El comando terminó con código {codigo}.')
    print('✓ Comando terminado correctamente')

## Paso 3 - Descargar y extraer el ZIP del proyecto

Se usa el ZIP de la rama `main` del repositorio. Este archivo no contiene `.env`, checkpoints ni ejecuciones anteriores.

In [ ]:
from zipfile import ZipFile

URL_ZIP = 'https://github.com/CaMunozS/topicos2/archive/refs/heads/main.zip'
CONTENIDO = Path('/content')
ARCHIVO_ZIP = CONTENIDO / 'topicos2-main.zip'

ejecutar(['curl', '--fail', '--location', '--show-error', '--output', ARCHIVO_ZIP, URL_ZIP])
print('ZIP descargado:', ARCHIVO_ZIP, f'({ARCHIVO_ZIP.stat().st_size / 1_000_000:.2f} MB)')

with ZipFile(ARCHIVO_ZIP) as zf:
    raices = {Path(nombre).parts[0] for nombre in zf.namelist() if nombre and not nombre.startswith('__MACOSX/')}
    print('Carpeta raíz del ZIP:', raices)
    if len(raices) != 1:
        raise RuntimeError('El ZIP debe tener exactamente una carpeta raíz.')
    zf.extractall(CONTENIDO)

DIRECTORIO_PROYECTO = CONTENIDO / raices.pop()
RUTA_REQUIREMENTS = DIRECTORIO_PROYECTO / 'requirements.txt'
if not RUTA_REQUIREMENTS.exists() or not (DIRECTORIO_PROYECTO / 'run.py').exists():
    raise FileNotFoundError('El ZIP no contiene requirements.txt y run.py en la raíz esperada.')

print('Proyecto:', DIRECTORIO_PROYECTO)
print('requirements:', RUTA_REQUIREMENTS)
print('Archivos:', sorted(p.name for p in DIRECTORIO_PROYECTO.iterdir() if p.is_file()))

## Paso 4 - Instalar dependencias

Se conserva el PyTorch CUDA que ya trae Colab. El `requirements.txt` del repositorio instala las bibliotecas restantes.

In [ ]:
ejecutar([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
ejecutar([sys.executable, '-m', 'pip', 'install', '-r', RUTA_REQUIREMENTS])

import transformers
import peft
import accelerate
print('\nVersiones instaladas:')
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('accelerate:', accelerate.__version__)
print('CUDA activa:', torch.cuda.is_available())

## Paso 5 - Comprobar secretos sin revelarlos

`HF_TOKEN` es opcional para el modelo público predeterminado; úsalo solo para modelos restringidos. `OPENAI_API_KEY` es obligatorio únicamente para el entrenamiento real. Guarda ambos valores en **Colab Secrets**; el notebook solo informa si están disponibles.

In [ ]:
def secreto_colab(nombre):
    valor = os.environ.get(nombre)
    if valor:
        return valor
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return None

HF_TOKEN = secreto_colab('HF_TOKEN')
OPENAI_API_KEY = secreto_colab('OPENAI_API_KEY')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print('HF_TOKEN disponible:', bool(HF_TOKEN))
print('OPENAI_API_KEY disponible:', bool(OPENAI_API_KEY))
print('Los valores nunca se imprimen ni se escriben en disco.')

## Paso 6 - Prueba completa del pipeline

Ejecuta `run.py --simulate`: no llama a OpenAI ni descarga el modelo de 4B. Sirve para validar datos, LoRA, checkpoints, dashboard y registros. Aunque hay GPU disponible, este modo usa un modelo diminuto de prueba.

In [ ]:
ejecutar([sys.executable, '-u', 'run.py', '--simulate'], cwd=DIRECTORIO_PROYECTO)

## Paso 7 - Entrenamiento real del modelo de 4B en GPU

La ejecución real usa `OPENAI_API_KEY`, descarga el modelo desde Hugging Face la primera vez y puede generar cargos. Para habilitarla, cambia el interruptor a `True`. Se limita inicialmente a una ronda y US$2; ajusta esos límites solo después de revisar la configuración.

In [ ]:
EJECUTAR_ENTRENAMIENTO_REAL = False
MAX_RONDAS = 1
PRESUPUESTO_USD = 2

if EJECUTAR_ENTRENAMIENTO_REAL:
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('Configura OPENAI_API_KEY en Colab Secrets antes de ejecutar el entrenamiento real.')
    ejecutar(
        [sys.executable, '-u', 'run.py', '--rounds', str(MAX_RONDAS), '--budget', str(PRESUPUESTO_USD)],
        cwd=DIRECTORIO_PROYECTO,
    )
else:
    print('Entrenamiento real no iniciado. Cambia EJECUTAR_ENTRENAMIENTO_REAL a True cuando estés listo.')

## Paso 8 - Ver resultados

Lista las corridas generadas y muestra el estado y resumen de la más reciente, si existe.

In [ ]:
import json

carpeta_runs = DIRECTORIO_PROYECTO / 'runs'
corridas = sorted([p for p in carpeta_runs.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime) if carpeta_runs.exists() else []
print('Corridas encontradas:', [p.name for p in corridas])
if corridas:
    ultima = corridas[-1]
    print('Última corrida:', ultima)
    for nombre in ['state.json', 'usage_summary.json']:
        archivo = ultima / nombre
        if archivo.exists():
            print(f'\n--- {nombre} ---')
            print(json.dumps(json.loads(archivo.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
else:
    print('Aún no hay corridas. Ejecuta la simulación o el entrenamiento real.')